# MITC 5-DOF vs the 6-DOF constrained element

OpenSG-TW carries **two** Reissner–Mindlin elements:

1. the **5-DOF eliminated-drilling element** `[w1,w2,w3,ω1,ω2]` of the prismatic
   (cross-sectional) pipeline, with the drilling rotation ω₃ eliminated through the
   in-plane symmetry and transverse-shear locking removed by **selective MITC**
   (Dvorkin–Bathe assumed-strain tying); and
2. the **6-DOF constrained element** `[w1,w2,w3,ω1,ω2,ω3]` of the tapered pipeline,
   with ω₃ kept independent, the symmetry enforced exactly by element-wise Lagrange
   multipliers, and the transverse shear **fully integrated**.

The production shear treatment is **full integration on the tapered segment**;
the **rings** retain the Dvorkin–Bathe $\gamma_{23}$ tie, which is exact on the
span-invariant strip (under span invariance $\gamma_{13}$ has no $w_i$
derivatives) and verified *inert* — full integration gives the same ring to
within 0.05 points on flat and curved sections over two decades of thickness.
Canonical MITC tying of the complete rows must **not** be used: it aliases the
algebraic drilling content of the six-parameter shear rows.
This notebook answers two questions with executed numbers: **(a)** how do the
two elements compare on a cross-section (ring SG), and **(b)** is the element
locking-free under full integration?  Benchmarks: the flat-walled **square** tube (the
stress case: the drilling elimination is singular on whole walls) and the smooth
**circle**, thin wall $t/R=0.02$, against 3-D solid references bundled in
`examples/data/benchmark/`.

In [1]:
import os, sys, io, contextlib, time
import numpy as np

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
MRS = os.path.join(REPO, 'mitc_rm_segment')
for p in (REPO, MRS):
    if p not in sys.path:
        sys.path.insert(0, p)
BENCH = os.path.join(REPO, 'examples', 'data', 'benchmark')

from boundary_from_yaml import extract
from segment_element import compute_k22
from segment_element_general import ring_general          # 5-DOF eliminated + MITC
from run_ring_indep import ring_indep                     # 6-DOF constrained
from solve_segment_jax import _material_by_section
import json

LBL = ['EA', 'GA2', 'GA3', 'GJ', 'EI2', 'EI3']
SCR = os.path.join(MRS, 'out', 'nb_5v6')
os.makedirs(SCR, exist_ok=True)

def ring_inputs(geom, mat):
    """Fresh geometry-tagged extraction of the thin aR=0.7 mesh -> L-ring arrays."""
    sub = 'taper_square' if geom == 'square' else 'taper_study'
    tg = 'thin_%s_aR070' % mat
    npz = os.path.join(SCR, '%s_%s.npz' % (geom, tg))
    with contextlib.redirect_stdout(io.StringIO()):
        extract(os.path.join(MRS, 'out', sub, 'meshes', 'shell_%s.yaml' % tg), npz)
    b = np.load(npz, allow_pickle=True)
    ax = int(b['axis']); cross = [j for j in range(3) if j != ax]
    rx = np.asarray(b['L_x']); rc = np.asarray(b['L_cells'])
    rs = np.asarray(b['L_subdom']); re3 = np.asarray(b['L_e3'])
    D_by, G_by = _material_by_section(json.loads(str(b['sections'])),
                                      json.loads(str(b['materials'])), center_ref=True)
    kr = compute_k22(rx[rc].mean(1), np.asarray(b['L_e2']), re3, rc)
    npzn = 'taper_square_solid_%s.npz' if geom == 'square' else 'taper_study_solid_%s.npz'
    s = np.load(os.path.join(BENCH, npzn % mat), allow_pickle=True)
    So = 0.5 * (s[tg + '_L'] + s[tg + '_L'].T)
    return (rx, rc, rs, re3, D_by, G_by, kr, ax, cross), So

## (a) Cross-section (ring SG): 5-DOF MITC vs 6-DOF, all shear treatments

Both elements solve the same one-element-deep wrapped strip (the exact prismatic
reduction of the surface element).  For the 6-DOF ring we run all three shear
treatments — full integration, tie-$\gamma_{23}$, tie-both — to expose how much the
MITC tying matters on a span-invariant configuration.

In [2]:
for geom in ('square', 'circle'):
    for mat in ('iso', 'm45'):
        args, So = ring_inputs(geom, mat)
        rows = []
        C5, _, _ = ring_general(*args, shear='mitc4_both')
        rows.append(('5-DOF elim + MITC', 0.5 * (C5 + C5.T)))
        rows.append(('6-DOF constr (g23-tied)', ring_indep(*args)))
        print('== %s thin %s : L ring, %%err vs solid ==' % (geom, mat))
        print('%-24s' % 'element' + ''.join('%8s' % l for l in LBL))
        for nm, S in rows:
            e = [100 * (S[i, i] - So[i, i]) / So[i, i] for i in range(6)]
            print('%-24s' % nm + ''.join('%+7.1f ' % v for v in e))
        print()

== square thin iso : L ring, %err vs solid ==
element                       EA     GA2     GA3      GJ     EI2     EI3
5-DOF elim + MITC          +0.0    -3.6    -3.7    +9.6    -0.0    -0.0 
6-DOF constr (g23-tied)    +0.0    -3.3    -3.3    -3.8    -0.0    -0.0 



== square thin m45 : L ring, %err vs solid ==
element                       EA     GA2     GA3      GJ     EI2     EI3
5-DOF elim + MITC          +0.8    +1.1    +0.9    +9.0    +0.5    +0.5 
6-DOF constr (g23-tied)    +0.8    +1.5    +1.5    +1.4    +0.5    +0.5 



== circle thin iso : L ring, %err vs solid ==
element                       EA     GA2     GA3      GJ     EI2     EI3
5-DOF elim + MITC          +0.2    +0.3    +0.1    +0.1    +0.2    +0.2 
6-DOF constr (g23-tied)    +0.2    +0.1    +0.1    +0.1    +0.2    +0.2 



== circle thin m45 : L ring, %err vs solid ==
element                       EA     GA2     GA3      GJ     EI2     EI3
5-DOF elim + MITC          +2.2    +3.1    +3.1    +3.0    +2.2    +2.3 
6-DOF constr (g23-tied)    +2.2    +3.0    +3.0    +3.0    +2.2    +2.2 



**Reading the table.** The two elements agree on extension, shear, and bending
within a fraction of a point.  Two systematic differences:

1. **Flat-wall ring torsion**: the 5-DOF element overshoots GJ by ~+9–10% on the
   square — its floored drilling reciprocal injects a small spurious prismatic
   torsion on the $C_{33}\equiv0$ walls.  The 6-DOF ring repairs it (−3.8% iso,
   +1.4% m45).  On the circle both are sub-percent.
2. Both elements use their production shear treatments ($\gamma_{23}$-tied on
   the ring); on the span-invariant strip the assumed field reproduces the true
   shear exactly, so the treatment is exact there by construction.

## (b) Is the element locking-free under full integration?

The classical locking probe: a prismatic isotropic circular tube at the **coarsest**
mesh, with the wall thinned tenfold below the benchmark range, against the
closed-form Timoshenko constants.  Locking errors grow without bound as $t\to0$ at
fixed mesh; a locking-free element shows **thickness-independent** errors.

In [3]:
import taper_study as ts
import run_indep as ri

print('Prismatic iso circle vs closed-form: 6-DOF (production) vs 5-DOF full (control):')
print('%-10s %-8s' % ('t/R', 'mesh') + ''.join('%8s' % l for l in LBL))
for tR in (0.02, 0.002):
    ts.THICK['thin'] = tR
    a = ts.ana_iso(1.0, tR)
    for (nc, nl) in ((24, 5), (48, 10)):
        mdir = os.path.join(SCR, 'lock_t%s_nc%d' % (str(tR).replace('.', 'p'), nc))
        ts.gen_case('thin', 'iso', 1.0, mesh_dir=mdir, nc=nc, nl=nl)
        S6d = ri.shell_solve_lagrange(ts.tag_of('thin', 'iso', 1.0), mdir, SCR)
        S5r = np.asarray(ts.shell_solve(ts.tag_of('thin', 'iso', 1.0), shear='full',
                                        mesh_dir=mdir, res_dir=SCR)[1])
        S5d = 0.5 * (S5r + S5r.T)
        for lbl, S in (('6-DOF prod', S6d), ('5-DOF full', S5d)):
            e = [100 * (S[i, i] - a[i]) / a[i] for i in range(6)]
            print('%-10s %2dx%-4d %-11s' % (tR, nc, nl, lbl) + ''.join('%+7.2f ' % v for v in e))
ts.THICK['thin'] = 0.02

Prismatic iso circle vs closed-form: 6-DOF (production) vs 5-DOF full (control):
t/R        mesh          EA     GA2     GA3      GJ     EI2     EI3


0.02       24x5    6-DOF prod   -0.29   -0.15   -0.15   -1.86   -1.35   -1.35 
0.02       24x5    5-DOF full   -0.29   +0.52   +0.04   -1.86   -1.34   -1.34 


0.02       48x10   6-DOF prod   -0.07   -0.03   -0.03   -0.46   -0.34   -0.34 
0.02       48x10   5-DOF full   -0.07   +0.17   +0.02   -0.46   -0.33   -0.33 


0.002      24x5    6-DOF prod   -0.29   -0.16   -0.16   -1.87   -1.35   -1.35 
0.002      24x5    5-DOF full   -0.29   +0.50   +0.03   -1.87   -1.35   -1.35 


0.002      48x10   6-DOF prod   -0.07   -0.04   -0.04   -0.47   -0.34   -0.34 
0.002      48x10   5-DOF full   -0.07   +0.13   +0.01   -0.47   -0.34   -0.34 


The errors are identical to two digits at $t/R=0.02$ and $t/R=0.002$ — no locking,
**for either element**: the 5-DOF control is equally clean under full integration.
The immunity is therefore a property of the *problem class*, not of the drilling
DOF: the SG fluctuation solve under section-strain load columns is not a
bending-dominated plate problem — the shear columns demand genuinely finite
transverse shear, and the near-inextensional bending/twist responses are
representable by the bilinear fields without parasitic constraints.  Neither
element needs MITC for the segment; the drilling DOF earns its keep on the
*taper degeneracy* (GA₃ −40% → −2%), not on locking.

## Summary

| context | 5-DOF eliminated + MITC | 6-DOF constrained |
|---|---|---|
| smooth ring | validated reference | identical (±0.2 pts) |
| flat-wall ring | GJ +9–10% overshoot | GJ repaired (−3.8/+1.4%) |
| tapered flat wall | GA₃ −24/−40% (degenerate) | GA₂=GA₃ −2.9/−1.8% |
| shear treatment | tie γ₂₃ (MITC) | rings: tie γ₂₃ (exact, inert); segment: full integration |
| shear locking | none in this problem class | none in this problem class |

The production tapered pipeline therefore uses the 6-DOF element everywhere
(rings and segment) with full shear integration — see the
{doc}`tapered-segment tutorial <taper_indep_omega3>`.